<a href="https://colab.research.google.com/github/MK-ayaz/Unsloth_Studio_Colab/blob/main/Telegram_bot_that_can_scrape_videos_of_tiktok_users_and_sending_to_the_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import sys
# Install libraries
!{sys.executable} -m pip install python-telegram-bot yt-dlp --quiet
print('Setup complete: python-telegram-bot and yt-dlp installed.')

Setup complete: python-telegram-bot and yt-dlp installed.


In [4]:
import yt_dlp
import os
import asyncio
from telegram import Update, InputMediaVideo
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from google.colab import userdata

# Securely load the token
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
except Exception:
    TELEGRAM_BOT_TOKEN = 'YOUR_BOT_TOKEN_HERE'

def get_tiktok_videos(url):
    ydl_opts = {
        'format': 'best',
        'quiet': True,
        'no_warnings': True,
        'extract_flat': True,
        'ignoreerrors': True,
        'geo_bypass': True,
        'playlist_items': '1-5000',
    }
    video_list = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        if info and 'entries' in info:
            for entry in info['entries']:
                if entry and 'url' in entry:
                    video_list.append(entry['url'])
        elif info and 'url' in info:
            video_list.append(info['url'])
    return video_list

def download_single_video(url):
    ydl_opts = {
        'format': 'best',
        'outtmpl': 'tiktok_%(id)s.%(ext)s',
        'quiet': True,
        'no_warnings': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    text = update.message.text
    if 'tiktok.com' in text:
        status_msg = await update.message.reply_text("🔍 *Analyzing Profile...*\nFetching all available video links.", parse_mode='Markdown')
        try:
            urls = await asyncio.to_thread(get_tiktok_videos, text)
            count = len(urls)
            if count == 0:
                await status_msg.edit_text("⚠️ No videos found. The profile might be private.")
                return

            await status_msg.edit_text(f"📦 *Found {count} video(s).*\nDownloading and grouping into albums...", parse_mode='Markdown')

            batch_size = 10
            for i in range(0, count, batch_size):
                batch_urls = urls[i:i + batch_size]
                media_group = []
                temp_files = []

                for url in batch_urls:
                    try:
                        file_path = await asyncio.to_thread(download_single_video, url)
                        if os.path.exists(file_path):
                            temp_files.append(file_path)
                            media_group.append(InputMediaVideo(open(file_path, 'rb'), caption=f"Video {len(temp_files)+i}/{count}"))
                    except Exception as e:
                        print(f"Error downloading {url}: {e}")

                if media_group:
                    try:
                        await update.message.reply_media_group(media=media_group)
                    except Exception as e:
                        await update.message.reply_text(f"❌ Batch Error: {e}")
                    finally:
                        for f in temp_files:
                            try:
                                os.remove(f)
                            except: pass

                await asyncio.sleep(2)

            await update.message.reply_text("✅ *All videos have been sent!*", parse_mode='Markdown')
        except Exception as e:
            await update.message.reply_text(f"❌ *Fatal Error:* `{e}`", parse_mode='Markdown')

bot_application = None

async def start(u: Update, c: ContextTypes.DEFAULT_TYPE):
    await u.message.reply_text("👋 *Welcome!*\nSend me a TikTok profile or video link to get started.", parse_mode='Markdown')

async def main():
    global bot_application
    if bot_application: await stop_bot()
    bot_application = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
    bot_application.add_handler(CommandHandler('start', start))
    bot_application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    await bot_application.initialize()
    await bot_application.updater.start_polling()
    await bot_application.start()
    print('Bot is running...')

async def stop_bot():
    global bot_application
    print('Attempting to stop bot...')
    if bot_application:
        try:
            if bot_application.updater.running:
                await bot_application.updater.stop()
            await bot_application.stop()
            await bot_application.shutdown()
        except Exception as e:
            print(f'Error during shutdown: {e}')
        finally:
            bot_application = None

    # Kill any dangling tasks
    tasks = [t for t in asyncio.all_tasks() if t is not asyncio.current_task()]
    for task in tasks:
        task.cancel()

    print('Bot stopped and tasks cleaned up.')

In [5]:
import asyncio
# Ensure the previous cell was run to define main()
try:
    await main()
except NameError:
    print('❌ Error: Please run the cell above this one first to define the bot functions.')

Bot is running...


In [ ]:
# Execute the improved stopping function to fully halt the bot
await stop_bot()

Attempting to stop bot...
Bot stopped and tasks cleaned up.
